In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [28]:
import torch

a = torch.tensor([1,2,3])
b = torch.zeros(3,4)
c = torch.ones(3,4)
d = torch.randn(5,3)

In [41]:
x = torch.tensor([2.0], requires_grad = True)
y = x ** 2
y.backward(retain_graph = True)
y.backward()
x.grad

tensor([8.])

In [43]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = x.to(device)

In [49]:
x = torch.randn(100,1)
y = 3 * x + 2

In [56]:
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(1,1)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(1,1)


    def forward(self,x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)

        return x

In [57]:
model = SimpleModel()

In [58]:
criterion = nn.MSELoss()

In [59]:
optimizer = torch.optim.SGD(model.parameters(), lr = 0.01)

In [61]:
for epoch in range(50):

    optimizer.zero_grad()

    outputs = model(x)

    loss = criterion(outputs,y)

    loss.backward()

    optimizer.step()

In [62]:
from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self, X, Y):
        self.X = X
        self.Y = Y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset, 
    batch_size=32,
    shuffle=True
)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1,8,3)
        self.relu = nn.ReLU()
        self.fc = nn.Linear(8*26*26,10)


    def forward(self,x):
        x = self.conv(x)
        x = self.relu(x)
        x = x.view(x.size(0),-1)
        x = self.fc(x)

        return

In [3]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random


In [2]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [4]:
# Load dataset
dataset = datasets.ImageFolder(
    root="/kaggle/input/images/images",
    transform=transform
)


# DataLoader
dataloader = DataLoader(
    dataset,
    batch_size=30,
    shuffle=True,
)

In [5]:
images, labels = next(iter(dataloader))

print("Batch image tensor shape:", images.shape)
print("Batch labels tensor shape:", labels.shape)

# Number of classes
num_classes = len(dataset.classes)

Batch image tensor shape: torch.Size([30, 3, 224, 224])
Batch labels tensor shape: torch.Size([30])


In [6]:
# ResNet Building Blocks
class ResidualBlock(nn.Module):
    """Basic Residual Block with skip connection"""
    def __init__(self, in_channels):
        super(ResidualBlock, self).__init__()
        
        # TODO: Layer 1

        self.layer1Conv = nn.Conv2d(in_channels,in_channels,3,padding=1,stride=1)
        self.layer1BatchNorm = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU()
        
        
        # TODO: Layer 2
        self.layer2Conv = nn.Conv2d(in_channels,in_channels,5,padding=2,stride=1)
        self.layer2BatchNorm = nn.BatchNorm2d(in_channels)
        
        
    def forward(self, x):
        identity = x
        
        # TODO: Layer 1
        x = self.layer1Conv(x)
        x = self.layer1BatchNorm(x)
        x = self.relu(x)
        
        # TODO: Layer 2
        x = self.layer2Conv(x)
        x = self.layer2BatchNorm(x)

        out = x + identity
        
        return out


In [9]:
class CustomResNet(nn.Module):
    """Custom ResNet architecture without nn.Sequential"""
    def __init__(self):
        super(CustomResNet, self).__init__()
        
        # TODO: Initial convolution layer
       
        self.initialConv = nn.Conv2d(3,64,7,padding=3,stride=2)
        self.initialBatchNorm = nn.BatchNorm2d(64)
        self.initialRelu = nn.ReLU()
        self.initialMaxPool = nn.MaxPool2d(kernel_size=3,stride=2,padding=1)
        
        # TODO: Layer 1

        self.layer1Residual = ResidualBlock(64)
        
        # TODO: Layer 1 conv

        self.layer1Conv = nn.Conv2d(64,32,4,padding=7,stride=1)
        self.layer1BatchNorm = nn.BatchNorm2d(32)
        self.layer1Relu = nn.ReLU()
        
        # TODO: Layer 2

        self.layer2Residual = ResidualBlock(32)
        
        # TODO: Layer 2 conv

        self.layer2Conv = nn.Conv2d(32,16,5,padding=2,stride=2)
        self.layer2BatchNorm = nn.BatchNorm2d(16)
        self.layer2Relu = nn.ReLU()

        # TODO: Layer 3

        self.layer3Residual = ResidualBlock(16)
        self.layer3BatchNorm = nn.BatchNorm2d(16)
        
        
        # TODO: Global average pooling and FC layer

        self.adaptiveAvgPool = nn.AdaptiveAvgPool2d((1,1))
        self.linear = nn.Linear(16,1)

    def forward(self, x):
        # TODO: Initial conv

        x = self.initialConv(x)
        x = self.initialBatchNorm(x)
        x = self.initialRelu(x)
        x = self.initialMaxPool(x)
        
        # TODO: Layer 1
        x = self.layer1Residual(x)

        # TODO: Layer 1 conv
        x = self.layer1Conv(x)
        x = self.layer1BatchNorm(x)
        x = self.layer1Relu(x)

        # TODO: Layer 2
        x = self.layer2Residual(x)
        
        # TODO: Layer 2 conv
        x = self.layer2Conv(x)
        x = self.layer2BatchNorm(x)
        x = self.layer2Relu(x)
    
        # TODO: Layer 3
        x = self.layer3Residual(x)
        x = self.layer3BatchNorm(x)
        # TODO: Global pooling and FC layer
        x = self.adaptiveAvgPool(x)
        x = torch.flatten(x, 1)
        x = self.linear(x)

        return x
    

In [12]:
# Initialize model
device = torch.device("cpu")

model = CustomResNet().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 10

print(f"\nTraining Custom ResNet with {sum(p.numel() for p in model.parameters())} parameters")
print(f"Device: {device}\n")

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.float().to(device)

        # Forward pass
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item() * images.size(0)
        
        predicted = (torch.sigmoid(outputs) > 0.5).long()
        total += labels.size(0)
        correct += (predicted == labels.long()).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100 * correct / total

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")


Training Custom ResNet with 238817 parameters
Device: cpu

Epoch [1/10] Loss: 0.6584 | Accuracy: 61.67%
Epoch [2/10] Loss: 0.4674 | Accuracy: 88.33%
Epoch [3/10] Loss: 0.3835 | Accuracy: 93.33%
Epoch [4/10] Loss: 0.3104 | Accuracy: 95.00%
Epoch [5/10] Loss: 0.3345 | Accuracy: 95.00%
Epoch [6/10] Loss: 0.2932 | Accuracy: 95.00%
Epoch [7/10] Loss: 0.2992 | Accuracy: 91.67%
Epoch [8/10] Loss: 0.2673 | Accuracy: 95.00%
Epoch [9/10] Loss: 0.2761 | Accuracy: 98.33%
Epoch [10/10] Loss: 0.2360 | Accuracy: 96.67%


**Section C Online**

In [4]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)  # For CPU
    torch.cuda.manual_seed(seed)  # For GPU (if used)
    torch.cuda.manual_seed_all(seed)  # For all GPUs
    torch.backends.cudnn.deterministic = True  # Ensure deterministic behavior
    torch.backends.cudnn.benchmark = False


set_seed(42)

In [5]:
# Image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [6]:
# Load dataset
dataset = datasets.ImageFolder(
    root="/kaggle/input/c-images/images",
    transform=transform
)

# DataLoader
dataloader = DataLoader(
    dataset,
    batch_size=30,
    shuffle=True,
)

In [7]:
class MySGDMomentum(Optimizer):
    def __init__(self,params,lr=1e-3,momentum=0.9):
        defaults = dict(lr=lr,momentum=momentum)
        super().__init__(params,defaults)

    @torch.no_grad()
    def step(self):
        loss = None

        for group in self.param_groups:
            lr = group["lr"]
            beta = group["momentum"]

            for p in group["params"]:
                grad = p.grad
                state = self.state[p]

                if "velocity" not in state:
                    state["velocity"] = torch.zeros_like(p)

                v = state["velocity"]
                v.mul_(beta).add_(grad)
                p.add_(-lr * v)
        return loss

In [ ]:
class MyAdam(Optimizer):
    def __init__(self,params,lr=1e-3,betas=(0.9,0.999),eps=1e-8):
        defaults = dict(lr=lr,betas=betas,eps=eps)
        super().__init__(params,defaults)

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            lr = group["lr"]
            beta1,beta2 = group["betas"]
            eps = group["eps"]

            for p in group["params"]:
                grad = p.grad
                state = self.state[p]

                if len(state) == 0:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(p)
                    state["exp_avg_sq"] = torch.zeros_like(p)

                m = state["exp_avg"]
                v = state["exp_avg_sq"]

                state["step"] += 1
                t = state["step"]

                m.mul_(beta1).add_(grad,alpha=1-beta1)
                v.mul_(beta2).addcmul_(grad,grad,value=1-beta2)

                m_hat = m / (1-beta1 ** t)
                v_hat = v / (1-beta2 ** t)

                P.add_(-lr* m_hat / (v_hat.sqrt() + eps))

In [8]:
images, labels = next(iter(dataloader))

print("Batch image tensor shape:", images.shape)
print("Batch labels tensor shape:", labels.shape)

# Number of classes (auto-detected)
num_classes = len(dataset.classes)

Batch image tensor shape: torch.Size([30, 3, 224, 224])
Batch labels tensor shape: torch.Size([30])


In [9]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()

        # TODO: Block 1
        self.block1Conv = nn.Conv2d(3,16,5,padding=2,stride=1)
        self.block1relu = nn.ReLU()
        self.block1maxpool = nn.MaxPool2d(kernel_size=2,stride=2,padding=0)
        # TODO: Block 2
        self.block2Conv = nn.Conv2d(16,32,19,padding=3,stride=1)
        self.block2relu = nn.ReLU()
        self.block2maxpool = nn.MaxPool2d(kernel_size=2,stride=2,padding=0)
        # TODO: Block 3
        self.block3Conv = nn.Conv2d(32,64,4,padding=8,stride=2)
        self.block3relu = nn.ReLU()
        self.block3maxpool = nn.MaxPool2d(kernel_size=2,stride=2,padding=0)
        # TODO: Block 4
        self.block4Linear = nn.Linear(64*16*16,128)
        self.block4relu = nn.ReLU(128)
        # TODO: Block 5
        self.block5Linear = nn.Linear(128,2)
        

    def forward(self, x):
        # TODO: Block 1
        x = self.block1Conv(x)
        x = self.block1relu(x)
        x = self.block1maxpool(x)
        # TODO: Block 2
        x = self.block2Conv(x)
        x = self.block2relu(x)
        x = self.block2maxpool(x)
        # TODO: Block 3
        x = self.block3Conv(x)
        x = self.block3relu(x)
        x = self.block3maxpool(x)
        # TODO: Block 4
        x = torch.flatten(x,1)
        x = self.block4Linear(x)
        x = self.block4relu(x)
        # TODO: Block 5
        x = self.block5Linear(x)

        return x

In [10]:
device = torch.device("cpu")

model = SimpleCNN(num_classes).to(device)

criterion = nn.CrossEntropyLoss()
#optimizer = optim.Adam(model.parameters(), lr=1e-3)
optimizer = MySGDMomentum(model.parameters(),lr=1e-3)

num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100 * correct / total

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

Epoch [1/5] Loss: 0.6967 | Accuracy: 50.00%
Epoch [2/5] Loss: 0.6940 | Accuracy: 50.00%
Epoch [3/5] Loss: 0.6889 | Accuracy: 50.00%
Epoch [4/5] Loss: 0.6833 | Accuracy: 50.00%
Epoch [5/5] Loss: 0.6762 | Accuracy: 51.67%


In [1]:
from torch.optim import Optimizer